In [2]:



import os
import re

#froot = r'Y:\DaVinci'
#froot = r'\\Icnlabs-invivo\d\VISUOMOTOR'
froots = [r'\\Icnlabs-invivo\e\VISUOMOTOR', r'\\Icnlabs-invivo\d\VISUOMOTOR', r'Y:\DaVinci']
outputs = [r'C:\Users\ICNLab\CaImAn_GV\caiman\ICNLAB\Drive_E_Census.txt', 
           r'C:\Users\ICNLab\CaImAn_GV\caiman\ICNLAB\Drive_D_Census.txt',
           r'C:\Users\ICNLab\CaImAn_GV\caiman\ICNLAB\server_census.txt']

for n in range(len(froots)):
    print(n)
    base_path = froots[n]

    # regex for yyyymmdd and FOV#_T#
    date_re = re.compile(r'^\d{8}$')
    fov_re = re.compile(r'^FOV\d+_T\d+$')

    folder_paths = []

    for root, dirs, files in os.walk(base_path):
        parts = os.path.normpath(root).split(os.sep)

        # need at least: base / anything / yyyymmdd / FOV#_T#
        if len(parts) < 3:
            continue

        if date_re.match(parts[-2]) and fov_re.match(parts[-1]):
            folder_paths.append(root)

    import os
    import re

    # pattern to locate the start of the meaningful subpath
    pattern = re.compile(r'(?:^|[\\/])([^\\/]+[\\/]\d{8}[\\/]FOV\d+_T\d+.*)$')

    normalized_paths = []

    for p in folder_paths:
        p_norm = os.path.normpath(p)
        p_norm = p_norm.replace('\\\\', '\\')
        match = pattern.search(p_norm)
        if match:
            normalized_paths.append(os.sep + match.group(1))

    # normalized_paths now contains the cleaned paths

    with open(outputs[n], 'w', encoding='utf-8') as f:
        for path in normalized_paths:
            f.write(path + '\n')



0


KeyboardInterrupt: 

In [5]:
print(len(normalized_paths))
print(normalized_paths[0:10])

188
['\\BL6J100.4LL\\20241104\\FOV1_T1', '\\BL6J102.5RR\\20241114\\FOV7_T2', '\\NCVF110.4LL\\20241121\\FOV1_T1', '\\NCVF110.4LL\\20241121\\FOV1_T2', '\\NCVF110.4LL\\20241121\\FOV2_T1', '\\NCVF110.4LL\\20241121\\FOV2_T2', '\\NCVF110.4LL\\20241121\\FOV3_T1', '\\NCVF110.4LL\\20241121\\FOV4_T1', '\\NCVF86.5L\\20241106\\FOV3_T1', '\\NCVF86.5L\\20241106\\FOV3_T2']


Load back text files

In [6]:
input_file_S = r'C:\Users\ICNLab\CaImAn_GV\caiman\ICNLAB\server_census.txt'
input_file_E = r'C:\Users\ICNLab\CaImAn_GV\caiman\ICNLAB\Drive_E_Census.txt'
input_file_D = r'C:\Users\ICNLab\CaImAn_GV\caiman\ICNLAB\Drive_D_Census.txt'


with open(input_file_S, 'r', encoding='utf-8') as f:
    folder_paths_S = [line.strip() for line in f if line.strip()]
    
with open(input_file_E, 'r', encoding='utf-8') as f:
    folder_paths_E = [line.strip() for line in f if line.strip()]

with open(input_file_D, 'r', encoding='utf-8') as f:
    folder_paths_D = [line.strip() for line in f if line.strip()]





Analyze overlap

In [7]:
import os
from collections import Counter

def normalize_list(paths):
    return [os.path.normpath(p) for p in paths]

S = normalize_list(folder_paths_S)
E = normalize_list(folder_paths_E)
D = normalize_list(folder_paths_D)

set_S = set(S)
set_E = set(E)
set_D = set(D)

only_E_not_S = sorted(set_E - set_S)
only_D_not_S = sorted(set_D - set_S)

only_ED_not_S = sorted((set_E | set_D) - set_S)

dupes_S = [p for p, c in Counter(S).items() if c > 1]
dupes_E = [p for p, c in Counter(E).items() if c > 1]
dupes_D = [p for p, c in Counter(D).items() if c > 1]

dupes_between_E_D = sorted(set_E & set_D)

print(f"S total: {len(S)}")
print(f"E total: {len(E)}")
print(f"D total: {len(D)}")

print(f"S internal duplicates: {len(dupes_S)}")
print(f"E internal duplicates: {len(dupes_E)}")
print(f"D internal duplicates: {len(dupes_D)}")

print(f"E only (not in S): {len(only_E_not_S)}")
print(f"D only (not in S): {len(only_D_not_S)}")
print(f"E ∪ D not in S: {len(only_ED_not_S)}")

print(f"E ∩ D duplicates: {len(dupes_between_E_D)}")


S total: 4296
E total: 477
D total: 188
S internal duplicates: 0
E internal duplicates: 0
D internal duplicates: 0
E only (not in S): 328
D only (not in S): 0
E ∪ D not in S: 328
E ∩ D duplicates: 0


Find all mouse IDs with data from June 2025 onward

In [3]:



import os
import re

froot = r'\\128.101.238.33\InVivo\DaVinci'
#froot = r'\\Icnlabs-invivo\d\VISUOMOTOR'
#froots = [r'\\Icnlabs-invivo\e\VISUOMOTOR', r'\\Icnlabs-invivo\d\VISUOMOTOR', r'Y:\DaVinci']
#outputs = [r'C:\Users\ICNLab\CaImAn_GV\caiman\ICNLAB\Drive_E_Census.txt', 
           #r'C:\Users\ICNLab\CaImAn_GV\caiman\ICNLAB\Drive_D_Census.txt',
           #r'C:\Users\ICNLab\CaImAn_GV\caiman\ICNLAB\server_census.txt']

output = r'C:\Users\ICNLab\CaImAn_GV\caiman\ICNLAB\server_census.txt'

base_path = froot

# regex for yyyymmdd and FOV#_T#
date_re = re.compile(r'^\d{8}$')
fov_re = re.compile(r'^FOV\d+_T\d+$')

folder_paths = []

for root, dirs, files in os.walk(base_path):
    parts = os.path.normpath(root).split(os.sep)

    # need at least: base / anything / yyyymmdd / FOV#_T#
    if len(parts) < 3:
        continue

    if date_re.match(parts[-2]) and fov_re.match(parts[-1]):
        folder_paths.append(root)

import os
import re

# pattern to locate the start of the meaningful subpath
pattern = re.compile(r'(?:^|[\\/])([^\\/]+[\\/]\d{8}[\\/]FOV\d+_T\d+.*)$')

normalized_paths = []

for p in folder_paths:
    p_norm = os.path.normpath(p)
    p_norm = p_norm.replace('\\\\', '\\')
    match = pattern.search(p_norm)
    if match:
        normalized_paths.append(os.sep + match.group(1))

# normalized_paths now contains the cleaned paths

with open(output, 'w', encoding='utf-8') as f:
    for path in normalized_paths:
        f.write(path + '\n')



In [4]:
from datetime import datetime

#normalized_paths alraedy defined

cutoff = datetime(2025, 6, 1)

ids_with_recent_data = set()

for path in normalized_paths:
    parts = path.strip("\\").split("\\")
    animal_id = parts[0]
    date_str = parts[1]

    date = datetime.strptime(date_str, "%Y%m%d")
    if date >= cutoff:
        ids_with_recent_data.add(animal_id)

print(sorted(ids_with_recent_data))


['BL6J113.8LL', 'BL6J154.3B', 'CCVF127.2R', 'CCVF127.3B', 'CCVF127.7B', 'CCVF127.8LL', 'CCVF140.6L', 'CCVF140.7R', 'CKCVF155.1L', 'CKCVF155.3L', 'CKCVF155.4R', 'CKCVF155.5B', 'CKCVF155.6LL', 'CKCVF155.7RR', 'CKCVF175.2R', 'CKCVF175.3B', 'NC162.1L', 'NC162.3B', 'NCVF128.3B', 'NCVF128.4L', 'NCVF128.6B', 'NF107.2R', 'NF115.12L', 'NF170.1L', 'NF170.2R', 'NF170.3B', 'NPCNF133.1L', 'NPCNF133.3L', 'NPCNF139.1L', 'NPCNF139.2R', 'NPCNF139.3B', 'NPCNF139.5R', 'NPCNF139.6B', 'NPCNF163.1L', 'NPCNF163.3B', 'NPCNF163.4L', 'NPCNF163.5R', 'NPCNF171.1L', 'NPCNF171.2R', 'NPCNF171.3B', 'NPCNF171.4LL', 'NPCNF171.6R', 'NPCNF171.7B', 'PC151.3L', 'PC173.2R', 'PC173.3B', 'SC142.2R', 'SCVF118.1L', 'SCVF118.2R', 'SCVF118.3B', 'SCVF118.4LL', 'SCVF96.10RR', 'SCVF96.6L', 'SCVF96.9LL', 'SF121.2R', 'SF121.4R', 'SF132.2R', 'SF132.3B', 'SF132.4L', 'SF132.6B', 'SMDATA', 'VC134.2R', 'VC134.3L', 'VC134.4R', 'VC136.2R', 'VC136.3B', 'VC136.4L', 'VC136.5R', 'VCNF150.1L', 'VCNF150.2R', 'VCNF150.3L', 'VCNF150.4R', 'VCNF150.6L

In [5]:
len(ids_with_recent_data)

83

In [9]:
#Already analyzed


Drive1 = ["NF170.1L",
    "NF170.3B",
    "NPCNF133.1L",
    "NPCNF139.1L",
    "NPCNF163.1L",
    "PC173.2R",
    "VCNF150.1L",
    "VCNF150.3L"]

Drive2 = ["NPCNF139.2R",
    "NPCNF139.5R",
    "NPCNF163.4L",
    "NPCNF171.1L",
    "NPCNF171.4LL"]


ids_not_in_drives = sorted(
    set(ids_with_recent_data) - set(Drive1) - set(Drive2)
)

print(ids_not_in_drives)
len(ids_not_in_drives)


['BL6J113.8LL', 'BL6J154.3B', 'CCVF127.2R', 'CCVF127.3B', 'CCVF127.7B', 'CCVF127.8LL', 'CCVF140.6L', 'CCVF140.7R', 'CKCVF155.1L', 'CKCVF155.3L', 'CKCVF155.4R', 'CKCVF155.5B', 'CKCVF155.6LL', 'CKCVF155.7RR', 'CKCVF175.2R', 'CKCVF175.3B', 'NC162.1L', 'NC162.3B', 'NCVF128.3B', 'NCVF128.4L', 'NCVF128.6B', 'NF107.2R', 'NF115.12L', 'NF170.2R', 'NPCNF133.3L', 'NPCNF139.3B', 'NPCNF139.6B', 'NPCNF163.3B', 'NPCNF163.5R', 'NPCNF171.2R', 'NPCNF171.3B', 'NPCNF171.6R', 'NPCNF171.7B', 'PC151.3L', 'PC173.3B', 'SC142.2R', 'SCVF118.1L', 'SCVF118.2R', 'SCVF118.3B', 'SCVF118.4LL', 'SCVF96.10RR', 'SCVF96.6L', 'SCVF96.9LL', 'SF121.2R', 'SF121.4R', 'SF132.2R', 'SF132.3B', 'SF132.4L', 'SF132.6B', 'SMDATA', 'VC134.2R', 'VC134.3L', 'VC134.4R', 'VC136.2R', 'VC136.3B', 'VC136.4L', 'VC136.5R', 'VCNF150.2R', 'VCNF150.4R', 'VCNF150.6LL', 'VF131.4LL', 'VF131.5L', 'VF131.6R', 'VF148.1L', 'VF148.2R', 'VF148.3B', 'VF148.4LL', 'VF148.5RR', 'VF167.1L', 'VF167.2R']


70